# Episode and Recurrence Features — `episode_recurrence_v1`

This notebook adds **secondary temporal context** to the frozen `point_event_v1` dataset.

The primary modeling unit remains **one candidate low-altitude point event**. This notebook does not collapse point events into incident rows. It:

- assigns each point event to a stable-aircraft episode using the frozen 450-second gap threshold;
- identifies singleton and multi-observation episodes;
- adds strictly backward-looking aircraft recurrence features;
- adds prior aircraft-behavior baselines and deviations;
- adds strictly backward-looking site-activity counts;
- preserves all 12,185 authoritative `point_event_v1` rows.

No anomaly or threat label is created in this notebook.


## Output contract

The notebook writes derived artifacts to:

```text
data/processed/point_event_v1/context_v1/
├── candidate_point_events_v1_with_episode_recurrence.csv
├── event_episode_mapping_v1.csv
├── point_event_context_v1_build_summary.json
├── point_event_context_v1_data_dictionary.csv
└── SHA256SUMS.txt
```

The authoritative input file is never overwritten.


## 1. Imports and project paths


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
from typing import Any

import numpy as np
import pandas as pd
import yaml

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

CONTEXT_VERSION = "episode_recurrence_v1"
RECURRENCE_WINDOWS_S = {
    "1h": 60 * 60,
    "24h": 24 * 60 * 60,
    "7d": 7 * 24 * 60 * 60,
    "30d": 30 * 24 * 60 * 60,
}
SITE_WINDOWS_S = {
    "1h": 60 * 60,
    "24h": 24 * 60 * 60,
    "7d": 7 * 24 * 60 * 60,
}


def find_project_root(start: Path) -> Path:
    """Find the nearest project directory containing the frozen point-event input and configuration."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        config_path = candidate / "config" / "point_event_v1.yaml"
        input_path = candidate / "data" / "processed" / "point_event_v1" / "candidate_point_events_v1.csv"
        if config_path.is_file() and input_path.is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate config/point_event_v1.yaml and "
        "data/processed/point_event_v1/candidate_point_events_v1.csv. "
        "Run this notebook from the Jupyter Notebook project directory or one of its subdirectories."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "config" / "point_event_v1.yaml"
POINT_EVENT_DIR = PROJECT_ROOT / "data" / "processed" / "point_event_v1"
INPUT_PATH = POINT_EVENT_DIR / "candidate_point_events_v1.csv"
INPUT_DICTIONARY_PATH = POINT_EVENT_DIR / "point_event_v1_data_dictionary.csv"
OUTPUT_DIR = POINT_EVENT_DIR / "context_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input:        {INPUT_PATH}")
print(f"Configuration:{CONFIG_PATH}")
print(f"Output:       {OUTPUT_DIR}")


## 2. Load and validate the frozen point-event configuration


In [ ]:
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

assert config["definition"]["version"] == "point_event_v1"
assert config["definition"]["status"] == "frozen"
assert config["secondary_context"]["collapse_events_into_episode_rows"] is False

EPISODE_GAP_THRESHOLD_S = int(config["secondary_context"]["episode_gap_threshold_s"])
EXPECTED_INPUT_ROWS = int(config["regression_expectations"]["qualifying_after_deduplication"])

print(json.dumps({
    "point_event_version": config["definition"]["version"],
    "context_version": CONTEXT_VERSION,
    "episode_gap_threshold_s": EPISODE_GAP_THRESHOLD_S,
    "recurrence_windows_s": RECURRENCE_WINDOWS_S,
    "site_windows_s": SITE_WINDOWS_S,
    "collapse_events_into_episode_rows": False,
}, indent=2))


## 3. Helper functions


In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Return the SHA-256 digest of a file."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def require_text(value: object, field_name: str) -> str:
    """Return a required pandas scalar or Python value as normalized text."""
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")

    if isinstance(value, bytes):
        text = value.decode("utf-8", errors="strict")
    else:
        text = str(value)

    text = text.strip()
    if not text or text.lower() in {"nan", "nat", "none", "<na>"}:
        raise ValueError(f"{field_name} cannot be empty or missing.")

    return text


def require_utc_timestamp(value: object, field_name: str) -> pd.Timestamp:
    """Return a required pandas scalar or Python value as a UTC Timestamp."""
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")

    if isinstance(value, pd.Timestamp):
        timestamp = value
    elif isinstance(value, datetime):
        timestamp = pd.Timestamp(value)
    elif isinstance(value, np.datetime64):
        timestamp = pd.Timestamp(value)
    elif isinstance(value, bytes):
        timestamp = pd.Timestamp(value.decode("utf-8", errors="strict"))
    else:
        # Converting through text avoids passing pandas' broad Scalar type
        # directly to pd.Timestamp and keeps static type checkers satisfied.
        timestamp = pd.Timestamp(str(value).strip())

    if pd.isna(timestamp):
        raise ValueError(f"{field_name} cannot be missing.")

    if timestamp.tzinfo is None:
        timestamp = timestamp.tz_localize("UTC")
    else:
        timestamp = timestamp.tz_convert("UTC")

    return timestamp


def make_episode_id(
    aircraft_key: object,
    episode_start_event_id: object,
    episode_start_time_utc: object,
) -> str:
    """Create a deterministic episode identifier without replacing the point-event identifier."""
    normalized_aircraft_key = require_text(aircraft_key, "aircraft_key")
    normalized_start_event_id = require_text(
        episode_start_event_id,
        "episode_start_event_id",
    )
    normalized_start_time = require_utc_timestamp(
        episode_start_time_utc,
        "episode_start_time_utc",
    )

    canonical = "|".join(
        [
            CONTEXT_VERSION,
            normalized_aircraft_key,
            normalized_start_event_id,
            normalized_start_time.isoformat(),
        ]
    )
    digest = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
    return f"EPV1-{digest[:16].upper()}"


def strict_prior_window_counts(times_ns: np.ndarray, window_s: int) -> np.ndarray:
    """Count events in [t-window, t), excluding the current timestamp and simultaneous peers."""
    left_current = np.searchsorted(times_ns, times_ns, side="left")
    lower = np.searchsorted(
        times_ns,
        times_ns - int(window_s * 1_000_000_000),
        side="left",
    )
    return (left_current - lower).astype("int32")


def add_aircraft_history_features(group: pd.DataFrame) -> pd.DataFrame:
    """Add strictly backward-looking recurrence and prior-behavior features for one aircraft."""
    group = group.sort_values(
        ["aircraft_event_time_utc", "snapshot_time_utc", "event_id"],
        kind="mergesort",
    ).copy()

    times_ns = group["aircraft_event_time_utc"].astype("int64").to_numpy()
    left_current = np.searchsorted(times_ns, times_ns, side="left")
    group["prior_event_count"] = left_current.astype("int32")

    for label, seconds in RECURRENCE_WINDOWS_S.items():
        group[f"events_previous_{label}"] = strict_prior_window_counts(times_ns, seconds)

    # The previous time must be strictly earlier. Events with identical event times do not see each other.
    unique_times = np.unique(times_ns)
    previous_by_time: dict[int, int | None] = {
        int(current): (int(unique_times[index - 1]) if index > 0 else None)
        for index, current in enumerate(unique_times)
    }
    previous_ns = np.array(
        [
            previous_by_time[int(current)]
            if previous_by_time[int(current)] is not None
            else np.nan
            for current in times_ns
        ],
        dtype="float64",
    )
    group["time_since_previous_event_s"] = (times_ns - previous_ns) / 1_000_000_000

    # Aggregate by exact event time before calculating expanding means. This prevents
    # same-time rows from contributing information to one another.
    value_columns = [
        "altitude_ft",
        "ground_speed_kt",
        "distance_miles",
        "vertical_speed_fpm",
    ]
    timestamp_aggregates = (
        group.assign(_event_ns=times_ns)
        .groupby("_event_ns", sort=True)
        .agg(
            _event_count=("event_id", "size"),
            **{f"_sum_{column}": (column, "sum") for column in value_columns},
        )
        .reset_index()
    )

    timestamp_aggregates["_prior_count"] = (
        timestamp_aggregates["_event_count"].cumsum().shift(fill_value=0)
    )

    for column in value_columns:
        timestamp_aggregates[f"_prior_sum_{column}"] = (
            timestamp_aggregates[f"_sum_{column}"].cumsum().shift(fill_value=0.0)
        )
        timestamp_aggregates[f"prior_mean_{column}"] = (
            timestamp_aggregates[f"_prior_sum_{column}"]
            / timestamp_aggregates["_prior_count"].replace(0, np.nan)
        )

    lookup = timestamp_aggregates.set_index("_event_ns")
    for column in value_columns:
        prior_column = f"prior_mean_{column}"
        group[prior_column] = [
            lookup.loc[int(event_ns), prior_column]
            for event_ns in times_ns
        ]
        group[f"{column}_deviation_from_prior_mean"] = (
            group[column] - group[prior_column]
        )

    return group


## 4. Load and validate the authoritative `point_event_v1` table


In [ ]:
STRING_COLUMNS = [
    "event_id",
    "event_definition_version",
    "aircraft_key",
    "aircraft_key_source",
    "flight_track_id",
    "hex",
    "callsign",
    "fr24_id",
    "source_type",
    "squawk",
    "deduplication_status",
    "source_filename",
]

events = pd.read_csv(
    INPUT_PATH,
    dtype={column: "string" for column in STRING_COLUMNS},
    parse_dates=["snapshot_time_utc", "aircraft_event_time_utc"],
)

required_columns = {
    "event_id",
    "event_definition_version",
    "candidate_event_flag",
    "aircraft_key",
    "snapshot_time_utc",
    "aircraft_event_time_utc",
    "altitude_ft",
    "ground_speed_kt",
    "vertical_speed_fpm",
    "distance_miles",
}
missing_columns = sorted(required_columns - set(events.columns))
if missing_columns:
    raise ValueError(f"Missing required point-event columns: {missing_columns}")

# Normalize identifier and timestamp columns before grouping. These conversions
# also narrow runtime values that pandas exposes to static type checkers as Scalar.
events["aircraft_key"] = events["aircraft_key"].map(
    lambda value: require_text(value, "aircraft_key")
).astype("string")
events["event_id"] = events["event_id"].map(
    lambda value: require_text(value, "event_id")
).astype("string")
events["aircraft_event_time_utc"] = pd.to_datetime(
    events["aircraft_event_time_utc"],
    utc=True,
    errors="raise",
)
events["snapshot_time_utc"] = pd.to_datetime(
    events["snapshot_time_utc"],
    utc=True,
    errors="raise",
)

assert len(events) == EXPECTED_INPUT_ROWS, (
    f"Expected {EXPECTED_INPUT_ROWS:,} point events but found {len(events):,}."
)
assert events["event_id"].is_unique
assert events["aircraft_key"].notna().all()
assert events["event_definition_version"].eq("point_event_v1").all()
assert events["candidate_event_flag"].eq(1).all()
assert str(events["aircraft_event_time_utc"].dt.tz) == "UTC"
assert str(events["snapshot_time_utc"].dt.tz) == "UTC"

INPUT_EVENT_IDS = frozenset(events["event_id"].astype(str))
INPUT_SHA256 = sha256_file(INPUT_PATH)

print(f"Input rows:             {len(events):,}")
print(f"Unique event IDs:       {events['event_id'].nunique():,}")
print(f"Unique aircraft keys:   {events['aircraft_key'].nunique():,}")
print(f"First event time:       {events['aircraft_event_time_utc'].min()}")
print(f"Last event time:        {events['aircraft_event_time_utc'].max()}")
print(f"Input SHA-256:          {INPUT_SHA256}")


## 5. Add calendar context

Calendar fields are derived directly from the event timestamp. They do not use future observations.


In [ ]:
events = events.sort_values(
    ["aircraft_key", "aircraft_event_time_utc", "snapshot_time_utc", "event_id"],
    kind="mergesort",
).reset_index(drop=True)

event_time = events["aircraft_event_time_utc"]
events["context_definition_version"] = CONTEXT_VERSION
events["event_year_utc"] = event_time.dt.year.astype("int16")
events["event_month_utc"] = event_time.dt.month.astype("int8")
events["event_day_utc"] = event_time.dt.day.astype("int8")
events["event_hour_utc"] = event_time.dt.hour.astype("int8")
events["event_day_of_week_utc"] = event_time.dt.dayofweek.astype("int8")
events["event_weekend_flag"] = event_time.dt.dayofweek.isin([5, 6]).astype("int8")

events[[
    "event_id",
    "aircraft_event_time_utc",
    "event_month_utc",
    "event_hour_utc",
    "event_day_of_week_utc",
    "event_weekend_flag",
]].head()


## 6. Assign point events to secondary aircraft episodes

Events for the same stable `aircraft_key` remain in the same episode when the event-time gap is no more than the configured 450 seconds. Episode assignment adds context but does not replace or aggregate point-event rows.


In [ ]:
events["gap_from_previous_same_aircraft_s"] = (
    events.groupby("aircraft_key", sort=False)["aircraft_event_time_utc"]
    .diff()
    .dt.total_seconds()
)

events["_new_episode_flag"] = (
    events["gap_from_previous_same_aircraft_s"].isna()
    | events["gap_from_previous_same_aircraft_s"].gt(EPISODE_GAP_THRESHOLD_S)
)

events["episode_sequence_number"] = (
    events.groupby("aircraft_key", sort=False)["_new_episode_flag"]
    .cumsum()
    .astype("int32")
)

events["_internal_episode_gap_s"] = events["gap_from_previous_same_aircraft_s"].where(
    ~events["_new_episode_flag"]
)

episode_group_columns = ["aircraft_key", "episode_sequence_number"]
episode_summary = (
    events.groupby(episode_group_columns, sort=False)
    .agg(
        episode_start_time_utc=("aircraft_event_time_utc", "min"),
        episode_end_time_utc=("aircraft_event_time_utc", "max"),
        episode_size=("event_id", "size"),
        episode_start_event_id=("event_id", "first"),
        episode_max_internal_gap_s=("_internal_episode_gap_s", "max"),
    )
    .reset_index()
)

episode_summary["episode_id"] = [
    make_episode_id(
        row.aircraft_key,
        row.episode_start_event_id,
        row.episode_start_time_utc,
    )
    for row in episode_summary.itertuples(index=False)
]

events = events.merge(
    episode_summary,
    on=episode_group_columns,
    how="left",
    validate="many_to_one",
)

events["episode_definition_version"] = CONTEXT_VERSION
events["episode_gap_threshold_s"] = EPISODE_GAP_THRESHOLD_S
events["episode_position"] = (
    events.groupby("episode_id", sort=False).cumcount().add(1).astype("int16")
)
events["multi_observation_episode_flag"] = events["episode_size"].gt(1).astype("int8")
events["duration_observable_flag"] = events["multi_observation_episode_flag"]

calculated_duration = (
    events["episode_end_time_utc"] - events["episode_start_time_utc"]
).dt.total_seconds()
events["observed_duration_s"] = calculated_duration.where(
    events["duration_observable_flag"].eq(1)
)
events["episode_max_internal_gap_s"] = events["episode_max_internal_gap_s"].where(
    events["duration_observable_flag"].eq(1)
)

events = events.drop(columns=["_new_episode_flag", "_internal_episode_gap_s"])

print(f"Episodes:                    {events['episode_id'].nunique():,}")
print(f"Singleton episodes:          {events.loc[events['episode_size'].eq(1), 'episode_id'].nunique():,}")
print(f"Multi-observation episodes:  {events.loc[events['episode_size'].gt(1), 'episode_id'].nunique():,}")
print(f"Events in multi episodes:    {int(events['multi_observation_episode_flag'].sum()):,}")
print(f"Maximum episode size:        {int(events['episode_size'].max()):,}")


## 7. Episode regression checks


In [ ]:
EXPECTED_EPISODE_RESULTS = {
    "input_events": 12_185,
    "unique_aircraft_keys": 2_035,
    "episodes": 12_168,
    "singleton_episodes": 12_158,
    "multi_observation_episodes": 10,
    "events_in_multi_observation_episodes": 27,
    "maximum_episode_size": 8,
}

ACTUAL_EPISODE_RESULTS = {
    "input_events": int(len(events)),
    "unique_aircraft_keys": int(events["aircraft_key"].nunique()),
    "episodes": int(events["episode_id"].nunique()),
    "singleton_episodes": int(
        events.loc[events["episode_size"].eq(1), "episode_id"].nunique()
    ),
    "multi_observation_episodes": int(
        events.loc[events["episode_size"].gt(1), "episode_id"].nunique()
    ),
    "events_in_multi_observation_episodes": int(
        events["multi_observation_episode_flag"].sum()
    ),
    "maximum_episode_size": int(events["episode_size"].max()),
}

for metric, expected in EXPECTED_EPISODE_RESULTS.items():
    actual = ACTUAL_EPISODE_RESULTS[metric]
    assert actual == expected, f"{metric}: expected {expected:,}, found {actual:,}"

pd.DataFrame([
    {
        "metric": metric,
        "expected": expected,
        "actual": ACTUAL_EPISODE_RESULTS[metric],
        "match": expected == ACTUAL_EPISODE_RESULTS[metric],
    }
    for metric, expected in EXPECTED_EPISODE_RESULTS.items()
])


## 8. Add strictly backward-looking aircraft recurrence features

For an event at time *t*, all recurrence windows use `[t − window, t)`. The current event, simultaneous peer events, and future events are excluded.

Prior means are also calculated only from strictly earlier observations of the same stable aircraft.


In [ ]:
aircraft_history_parts: list[pd.DataFrame] = []

for _, aircraft_events in events.groupby("aircraft_key", sort=False, dropna=False):
    aircraft_history_parts.append(add_aircraft_history_features(aircraft_events))

events = (
    pd.concat(aircraft_history_parts, ignore_index=True)
    .sort_values(
        ["aircraft_event_time_utc", "aircraft_key", "event_id"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

for label in RECURRENCE_WINDOWS_S:
    events[f"recurrence_{label}_flag"] = (
        events[f"events_previous_{label}"].gt(0).astype("int8")
    )

events["first_observed_aircraft_event_flag"] = (
    events["prior_event_count"].eq(0).astype("int8")
)

recurrence_summary = pd.DataFrame([
    {
        "window": label,
        "events_with_prior_appearance": int(events[f"recurrence_{label}_flag"].sum()),
        "percentage": float(events[f"recurrence_{label}_flag"].mean() * 100),
    }
    for label in RECURRENCE_WINDOWS_S
])
recurrence_summary


## 9. Add strictly backward-looking site activity

Site counts use all candidate point events occurring before the current timestamp. Events with the exact same timestamp do not count one another.


In [ ]:
events = events.sort_values(
    ["aircraft_event_time_utc", "aircraft_key", "event_id"],
    kind="mergesort",
).reset_index(drop=True)

site_times_ns = events["aircraft_event_time_utc"].astype("int64").to_numpy()
for label, seconds in SITE_WINDOWS_S.items():
    events[f"site_events_previous_{label}"] = strict_prior_window_counts(
        site_times_ns,
        seconds,
    )

events[[
    "event_id",
    "aircraft_event_time_utc",
    "site_events_previous_1h",
    "site_events_previous_24h",
    "site_events_previous_7d",
]].head()


## 10. Validate row preservation, chronology, and leakage controls


In [ ]:
assert len(events) == EXPECTED_INPUT_ROWS
assert events["event_id"].is_unique
assert frozenset(events["event_id"].astype(str)) == INPUT_EVENT_IDS
assert events["episode_id"].notna().all()
assert events["episode_size"].ge(1).all()
assert events["episode_position"].ge(1).all()
assert events["episode_position"].le(events["episode_size"]).all()
assert events["episode_max_internal_gap_s"].dropna().le(EPISODE_GAP_THRESHOLD_S).all()
assert events.loc[events["episode_size"].eq(1), "observed_duration_s"].isna().all()
assert events.loc[events["episode_size"].gt(1), "observed_duration_s"].notna().all()
assert events["prior_event_count"].ge(events["events_previous_30d"]).all()
assert events["events_previous_30d"].ge(events["events_previous_7d"]).all()
assert events["events_previous_7d"].ge(events["events_previous_24h"]).all()
assert events["events_previous_24h"].ge(events["events_previous_1h"]).all()
assert int(events["first_observed_aircraft_event_flag"].sum()) == int(events["aircraft_key"].nunique())

# A first-observed aircraft event cannot have a prior mean or elapsed-time value.
first_events = events["first_observed_aircraft_event_flag"].eq(1)
assert events.loc[first_events, "time_since_previous_event_s"].isna().all()
for column in [
    "prior_mean_altitude_ft",
    "prior_mean_ground_speed_kt",
    "prior_mean_distance_miles",
    "prior_mean_vertical_speed_fpm",
]:
    assert events.loc[first_events, column].isna().all()

EXPECTED_RECURRENCE_RESULTS = {
    "events_with_prior_1h": 151,
    "events_with_prior_24h": 2_738,
    "events_with_prior_7d": 5_445,
    "events_with_prior_30d": 8_058,
}
ACTUAL_RECURRENCE_RESULTS = {
    "events_with_prior_1h": int(events["recurrence_1h_flag"].sum()),
    "events_with_prior_24h": int(events["recurrence_24h_flag"].sum()),
    "events_with_prior_7d": int(events["recurrence_7d_flag"].sum()),
    "events_with_prior_30d": int(events["recurrence_30d_flag"].sum()),
}
for metric, expected in EXPECTED_RECURRENCE_RESULTS.items():
    actual = ACTUAL_RECURRENCE_RESULTS[metric]
    assert actual == expected, f"{metric}: expected {expected:,}, found {actual:,}"

print("All row-preservation, episode, chronology, and leakage-control checks passed.")


## 11. Review the rare multi-observation episodes


In [ ]:
multi_episode_columns = [
    "episode_id",
    "aircraft_key",
    "event_id",
    "aircraft_event_time_utc",
    "episode_position",
    "episode_size",
    "gap_from_previous_same_aircraft_s",
    "altitude_ft",
    "ground_speed_kt",
    "distance_miles",
    "source_type",
]

events.loc[
    events["multi_observation_episode_flag"].eq(1),
    multi_episode_columns,
].sort_values(["episode_id", "episode_position"])


## 12. Define output columns and mappings


In [ ]:
INTERNAL_COLUMNS = {
    "episode_start_event_id",
}

# Keep all authoritative point-event fields plus all derived context fields.
output_events = events.drop(columns=[column for column in INTERNAL_COLUMNS if column in events.columns]).copy()

mapping_columns = [
    "event_id",
    "aircraft_key",
    "aircraft_event_time_utc",
    "episode_definition_version",
    "episode_gap_threshold_s",
    "episode_id",
    "episode_sequence_number",
    "episode_position",
    "episode_size",
    "episode_start_time_utc",
    "episode_end_time_utc",
    "gap_from_previous_same_aircraft_s",
    "episode_max_internal_gap_s",
    "observed_duration_s",
    "duration_observable_flag",
    "multi_observation_episode_flag",
]
episode_mapping = output_events[mapping_columns].copy()

assert len(output_events) == len(events)
assert len(episode_mapping) == len(events)
assert episode_mapping["event_id"].is_unique

print(f"Context-enriched event rows: {len(output_events):,}")
print(f"Episode mapping rows:        {len(episode_mapping):,}")
print(f"Total output columns:        {len(output_events.columns):,}")


## 13. Build the combined data dictionary


In [ ]:
NEW_COLUMN_DESCRIPTIONS = {
    "context_definition_version": "Version of the secondary episode and recurrence feature process.",
    "event_year_utc": "UTC calendar year of the aircraft event timestamp.",
    "event_month_utc": "UTC calendar month of the aircraft event timestamp, from 1 through 12.",
    "event_day_utc": "UTC day of month of the aircraft event timestamp.",
    "event_hour_utc": "UTC hour of day of the aircraft event timestamp, from 0 through 23.",
    "event_day_of_week_utc": "UTC weekday where Monday is 0 and Sunday is 6.",
    "event_weekend_flag": "1 when the UTC event date is Saturday or Sunday.",
    "gap_from_previous_same_aircraft_s": "Seconds since the immediately preceding event for the same stable aircraft; missing for its first observed event.",
    "episode_sequence_number": "Chronological episode number within the stable aircraft key.",
    "episode_definition_version": "Version of the episode-assignment rule.",
    "episode_gap_threshold_s": "Maximum same-aircraft event-time gap allowed within one episode.",
    "episode_id": "Deterministic identifier for the secondary aircraft episode.",
    "episode_position": "One-based position of the point event within its episode.",
    "episode_size": "Number of point events assigned to the episode.",
    "episode_start_time_utc": "Earliest aircraft event time in the episode.",
    "episode_end_time_utc": "Latest aircraft event time in the episode.",
    "episode_max_internal_gap_s": "Largest within-episode event-time gap; missing for singleton episodes.",
    "observed_duration_s": "Episode end minus start time in seconds; missing when duration is not observable from a singleton.",
    "duration_observable_flag": "1 when an episode contains more than one point event.",
    "multi_observation_episode_flag": "1 when the point event belongs to an episode containing more than one observation.",
    "prior_event_count": "Number of strictly earlier point events for the same stable aircraft.",
    "events_previous_1h": "Same-aircraft point events in the strictly prior one-hour window.",
    "events_previous_24h": "Same-aircraft point events in the strictly prior 24-hour window.",
    "events_previous_7d": "Same-aircraft point events in the strictly prior seven-day window.",
    "events_previous_30d": "Same-aircraft point events in the strictly prior 30-day window.",
    "time_since_previous_event_s": "Seconds since the most recent strictly earlier event for the same stable aircraft.",
    "prior_mean_altitude_ft": "Mean altitude of strictly earlier point events for the same stable aircraft.",
    "altitude_ft_deviation_from_prior_mean": "Current altitude minus the same aircraft's strictly prior mean altitude.",
    "prior_mean_ground_speed_kt": "Mean ground speed of strictly earlier point events for the same stable aircraft.",
    "ground_speed_kt_deviation_from_prior_mean": "Current ground speed minus the same aircraft's strictly prior mean ground speed.",
    "prior_mean_distance_miles": "Mean PANC distance of strictly earlier point events for the same stable aircraft.",
    "distance_miles_deviation_from_prior_mean": "Current distance minus the same aircraft's strictly prior mean distance.",
    "prior_mean_vertical_speed_fpm": "Mean vertical speed of strictly earlier point events for the same stable aircraft.",
    "vertical_speed_fpm_deviation_from_prior_mean": "Current vertical speed minus the same aircraft's strictly prior mean vertical speed.",
    "recurrence_1h_flag": "1 when the same stable aircraft appeared in the strictly prior one-hour window.",
    "recurrence_24h_flag": "1 when the same stable aircraft appeared in the strictly prior 24-hour window.",
    "recurrence_7d_flag": "1 when the same stable aircraft appeared in the strictly prior seven-day window.",
    "recurrence_30d_flag": "1 when the same stable aircraft appeared in the strictly prior 30-day window.",
    "first_observed_aircraft_event_flag": "1 for the earliest observed point event of each stable aircraft key.",
    "site_events_previous_1h": "All candidate point events at the site in the strictly prior one-hour window.",
    "site_events_previous_24h": "All candidate point events at the site in the strictly prior 24-hour window.",
    "site_events_previous_7d": "All candidate point events at the site in the strictly prior seven-day window.",
}

if INPUT_DICTIONARY_PATH.exists():
    input_dictionary = pd.read_csv(INPUT_DICTIONARY_PATH)
else:
    input_dictionary = pd.DataFrame(columns=["column_name", "logical_type", "description"])

new_dictionary_rows = []
for column, description in NEW_COLUMN_DESCRIPTIONS.items():
    if column not in output_events.columns:
        continue
    dtype = str(output_events[column].dtype)
    new_dictionary_rows.append({
        "column_name": column,
        "logical_type": dtype,
        "description": description,
    })

context_dictionary = pd.concat(
    [input_dictionary, pd.DataFrame(new_dictionary_rows)],
    ignore_index=True,
).drop_duplicates(subset=["column_name"], keep="last")

# Keep dictionary order aligned with the enriched output table.
column_order = {column: index for index, column in enumerate(output_events.columns)}
context_dictionary = (
    context_dictionary[context_dictionary["column_name"].isin(output_events.columns)]
    .assign(_order=lambda frame: frame["column_name"].map(column_order))
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

assert set(context_dictionary["column_name"]) == set(output_events.columns)
context_dictionary.tail(15)


## 14. Save outputs, checksums, and build summary


In [ ]:
ENRICHED_OUTPUT_PATH = OUTPUT_DIR / "candidate_point_events_v1_with_episode_recurrence.csv"
EPISODE_MAPPING_PATH = OUTPUT_DIR / "event_episode_mapping_v1.csv"
BUILD_SUMMARY_PATH = OUTPUT_DIR / "point_event_context_v1_build_summary.json"
DATA_DICTIONARY_PATH = OUTPUT_DIR / "point_event_context_v1_data_dictionary.csv"
CHECKSUMS_PATH = OUTPUT_DIR / "SHA256SUMS.txt"

output_events.to_csv(ENRICHED_OUTPUT_PATH, index=False)
episode_mapping.to_csv(EPISODE_MAPPING_PATH, index=False)
context_dictionary.to_csv(DATA_DICTIONARY_PATH, index=False)

regression_checks = []
for metric, expected in {**EXPECTED_EPISODE_RESULTS, **EXPECTED_RECURRENCE_RESULTS}.items():
    actual = (
        ACTUAL_EPISODE_RESULTS[metric]
        if metric in ACTUAL_EPISODE_RESULTS
        else ACTUAL_RECURRENCE_RESULTS[metric]
    )
    regression_checks.append({
        "metric": metric,
        "expected": int(expected),
        "actual": int(actual),
        "match": bool(expected == actual),
    })

build_summary: dict[str, Any] = {
    "definitions": {
        "point_event_version": config["definition"]["version"],
        "context_version": CONTEXT_VERSION,
        "episode_gap_threshold_s": EPISODE_GAP_THRESHOLD_S,
        "recurrence_windows_s": RECURRENCE_WINDOWS_S,
        "site_windows_s": SITE_WINDOWS_S,
        "point_events_collapsed": False,
        "future_information_used": False,
    },
    "build": {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "project_root": str(PROJECT_ROOT),
        "configuration_file": str(CONFIG_PATH.relative_to(PROJECT_ROOT)),
        "input_file": str(INPUT_PATH.relative_to(PROJECT_ROOT)),
        "input_sha256": INPUT_SHA256,
    },
    "input": {
        "point_event_rows": int(EXPECTED_INPUT_ROWS),
        "unique_aircraft_keys": int(output_events["aircraft_key"].nunique()),
        "first_event_time_utc": output_events["aircraft_event_time_utc"].min().isoformat(),
        "last_event_time_utc": output_events["aircraft_event_time_utc"].max().isoformat(),
    },
    "episodes": {
        "episode_count": int(output_events["episode_id"].nunique()),
        "singleton_episode_count": int(
            output_events.loc[output_events["episode_size"].eq(1), "episode_id"].nunique()
        ),
        "multi_observation_episode_count": int(
            output_events.loc[output_events["episode_size"].gt(1), "episode_id"].nunique()
        ),
        "events_in_multi_observation_episodes": int(
            output_events["multi_observation_episode_flag"].sum()
        ),
        "maximum_episode_size": int(output_events["episode_size"].max()),
    },
    "recurrence": {
        "first_observed_aircraft_events": int(
            output_events["first_observed_aircraft_event_flag"].sum()
        ),
        "events_with_prior_1h": int(output_events["recurrence_1h_flag"].sum()),
        "events_with_prior_24h": int(output_events["recurrence_24h_flag"].sum()),
        "events_with_prior_7d": int(output_events["recurrence_7d_flag"].sum()),
        "events_with_prior_30d": int(output_events["recurrence_30d_flag"].sum()),
    },
    "outputs": {
        "enriched_event_rows": int(len(output_events)),
        "episode_mapping_rows": int(len(episode_mapping)),
        "enriched_event_columns": int(len(output_events.columns)),
    },
    "regression_checks": regression_checks,
}

with BUILD_SUMMARY_PATH.open("w", encoding="utf-8") as handle:
    json.dump(build_summary, handle, indent=2)
    handle.write("\n")

files_to_hash = [
    ENRICHED_OUTPUT_PATH,
    EPISODE_MAPPING_PATH,
    BUILD_SUMMARY_PATH,
    DATA_DICTIONARY_PATH,
]
with CHECKSUMS_PATH.open("w", encoding="utf-8") as handle:
    for path in files_to_hash:
        handle.write(f"{sha256_file(path)}  {path.name}\n")

print("Saved:")
for path in [*files_to_hash, CHECKSUMS_PATH]:
    print(f"  {path.relative_to(PROJECT_ROOT)}")


## 15. Final validation summary


In [ ]:
final_summary = pd.DataFrame([
    {"metric": "Point events retained", "value": f"{len(output_events):,}"},
    {"metric": "Unique aircraft keys", "value": f"{output_events['aircraft_key'].nunique():,}"},
    {"metric": "Episodes", "value": f"{output_events['episode_id'].nunique():,}"},
    {
        "metric": "Singleton episodes",
        "value": f"{output_events.loc[output_events['episode_size'].eq(1), 'episode_id'].nunique():,}",
    },
    {
        "metric": "Multi-observation episodes",
        "value": f"{output_events.loc[output_events['episode_size'].gt(1), 'episode_id'].nunique():,}",
    },
    {"metric": "Events with prior appearance in 1 hour", "value": f"{output_events['recurrence_1h_flag'].sum():,}"},
    {"metric": "Events with prior appearance in 24 hours", "value": f"{output_events['recurrence_24h_flag'].sum():,}"},
    {"metric": "Events with prior appearance in 7 days", "value": f"{output_events['recurrence_7d_flag'].sum():,}"},
    {"metric": "Events with prior appearance in 30 days", "value": f"{output_events['recurrence_30d_flag'].sum():,}"},
    {"metric": "Output columns", "value": f"{len(output_events.columns):,}"},
])

final_summary


## Completion criterion

This notebook is complete when all cells run without errors and the final output retains exactly **12,185 unique point events**. Episodes and recurrence are context features only; the primary row remains the candidate low-altitude point event.
